# Car Detection Model Test

This notebook tests the car detection model that was saved by the `car_detection.ipynb` notebook. 
It loads the model from the `Car Detection/saved_model/ssd_mobilenet_v2_detector` directory, 
loads test images from URLs, performs detection, and displays the results with green bounding boxes 
and labels on top of the boxes.

## 1. Install and Import Dependencies

In [ ]:
# Install necessary libraries (if running in a new environment and not already installed)
!pip install -q tensorflow opencv-python-headless matplotlib Pillow

In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time

from six import BytesIO
from six.moves.urllib.request import urlopen
import os

## 2. Define Model Path and Load Saved Model

In [ ]:
MODEL_SAVE_PATH = "Car Detection/saved_model/ssd_mobilenet_v2_detector"

if not os.path.exists(MODEL_SAVE_PATH):
    print(f"Model not found at {MODEL_SAVE_PATH}.")
    print("Please run the `car_detection.ipynb` notebook first to download and save the model.")
    # You could raise an error here or stop execution if preferred.
    loaded_model = None
else:
    print(f"Loading model from {MODEL_SAVE_PATH}...")
    loaded_model = tf.saved_model.load(MODEL_SAVE_PATH)
    print("Model loaded successfully!")
    # The loaded_model is a SavedModel object that can be called directly for inference:
    # e.g., results = loaded_model(image_tensor)

## 3. Helper Functions (Adapted from previous notebook)

In [ ]:
def load_image_into_numpy_array(path_or_url):
    """Load an image from a file path or URL into a numpy array."""
    image = None
    if path_or_url.startswith('http'):
        try:
            response = urlopen(path_or_url)
            image_data = response.read()
            image_data = BytesIO(image_data)
            image = Image.open(image_data)
        except Exception as e:
            print(f"Error opening URL {path_or_url}: {e}")
            return None
    else:
        if not os.path.exists(path_or_url):
            print(f"Error: Image path not found {path_or_url}")
            return None
        image_data = tf.io.gfile.GFile(path_or_url, 'rb').read()
        image = Image.open(BytesIO(image_data))

    (im_width, im_height) = image.size
    return np.array(image.getdata()).reshape(
        (im_height, im_width, 3)).astype(np.uint8)

In [ ]:
def run_detector(detector_model, image_np):
    """Runs inference on a single image using the loaded SavedModel."""
    if detector_model is None:
        print("Detector model is not loaded. Cannot run inference.")
        return None
        
    image_tensor = tf.convert_to_tensor(image_np)[tf.newaxis, ...]
    
    start_time = time.time()
    # The loaded SavedModel can be called directly.
    # It typically returns a dictionary of tensors.
    result = detector_model(image_tensor)
    end_time = time.time()
    
    print(f"Inference time: {end_time - start_time:.2f}s")
    
    # Convert tensors in the result to numpy arrays
    result_np = {key: value.numpy() for key, value in result.items()}
    return result_np

In [ ]:
# COCO 2017 dataset category names and IDs (for label mapping)
COCO17_HUMAN_READABLE_NAMES = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

def draw_bounding_boxes_with_labels_on_top(image_np, results, min_score_thresh=0.5, target_class_names=['car', 'truck', 'bus']):
    """Draws green bounding boxes with class labels on top of the boxes."""
    if results is None:
        print("No results to draw.")
        return image_np
        
    image_with_boxes = image_np.copy()
    boxes = results['detection_boxes'][0]  # Assuming batch size of 1
    classes = results['detection_classes'][0].astype(int)
    scores = results['detection_scores'][0]
    
    im_height, im_width, _ = image_np.shape
    num_detections = 0
    
    # Map target class names to their IDs (COCO IDs are 1-indexed)
    target_class_ids = []
    for name in target_class_names:
        try:
            target_class_ids.append(COCO17_HUMAN_READABLE_NAMES.index(name.lower()) + 1)
        except ValueError:
            print(f"Warning: Class name '{name}' not found in COCO names. It will be ignored.")

    for i in range(boxes.shape[0]):
        if scores[i] >= min_score_thresh and classes[i] in target_class_ids:
            num_detections += 1
            ymin, xmin, ymax, xmax = tuple(boxes[i])
            (left, right, top, bottom) = (xmin * im_width, xmax * im_width,
                                          ymin * im_height, ymax * im_height)
            
            # Draw green rectangle
            cv2.rectangle(image_with_boxes, (int(left), int(top)), (int(right), int(bottom)), (0, 255, 0), 2)
            
            # Prepare label text
            class_name = "Unknown"
            if 0 < classes[i] <= len(COCO17_HUMAN_READABLE_NAMES):
                 class_name = COCO17_HUMAN_READABLE_NAMES[classes[i]-1]
            label = f"{class_name}: {scores[i]:.2f}"
            
            # Get text size to position it above the box
            (text_width, text_height), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            
            # Position label above the box
            label_ymin = max(top, text_height + 10) # Ensure label is not cut off at image top
            cv2.rectangle(image_with_boxes, (int(left), int(label_ymin - text_height - baseline)), 
                          (int(left + text_width), int(label_ymin)), (0, 255, 0), cv2.FILLED) # Background for text
            cv2.putText(image_with_boxes, label, (int(left), int(label_ymin - baseline)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1) # Black text on green background
                        
    print(f"Detected {num_detections} relevant objects.")
    return image_with_boxes

## 4. Define Test Images and Run Detection

In [ ]:
# Test image URLs from Unsplash
TEST_IMAGE_URLS = [
    "https://images.unsplash.com/photo-1738105525030-8d4664ba6abe?fm=jpg&q=60&w=3000&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8Mnx8c3RyZWV0JTIwdHJhZmZpYyUyMGNhcnN8ZW58MHx8MHx8fDA%3D",
    "https://images.unsplash.com/photo-1653399889170-6e5048f6588d?fm=jpg&q=60&w=3000&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8NHx8c3RyZWV0JTIwdHJhZmZpYyUyMGNhcnN8ZW58MHx8MHx8fDA%3D",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e4/Cars_in_traffic_in_Auckland%2C_New_Zealand_-_copyright-free_photo_released_to_public_domain.jpg/1280px-Cars_in_traffic_in_Auckland%2C_New_Zealand_-_copyright-free_photo_released_to_public_domain.jpg" # Previous default
]

if not TEST_IMAGE_URLS:
    print("No test image URLs provided. Please add some to TEST_IMAGE_URLS.")

if loaded_model is None:
    print("Skipping detection as model is not loaded.")
else:
    for image_url in TEST_IMAGE_URLS:
        print(f"\nProcessing image: {image_url}")
        image_np = load_image_into_numpy_array(image_url)
        
        if image_np is not None:
            plt.figure(figsize=(15, 10))
            plt.subplot(1, 2, 1)
            plt.title("Original Image")
            plt.imshow(image_np)
            plt.axis("off")

            # Run detection
            results = run_detector(loaded_model, image_np)
            
            # Draw bounding boxes for cars, trucks, buses with labels on top
            image_with_boxes = draw_bounding_boxes_with_labels_on_top(
                image_np, results, min_score_thresh=0.4, target_class_names=['car', 'truck', 'bus']
            )
            
            plt.subplot(1, 2, 2)
            plt.title("Detections (Model from File)")
            plt.imshow(image_with_boxes)
            plt.axis("off")
            plt.show()
        else:
            print(f"Could not load image {image_url}, skipping.")

## Notes

- Ensure that the `car_detection.ipynb` notebook has been run first to save the model to the `Car Detection/saved_model/ssd_mobilenet_v2_detector` directory.
- You can add more image URLs to the `TEST_IMAGE_URLS` list to test with different images.